In [ ]:
import pandas as pd
import numpy as np
from scipy.stats import zscore
import os

# Load your dataset
df = pd.read_csv(r'C:\Users\hp\Pictures\10 Academy\solar-challenge-week1\solar-challenge-week1\data\data\togo-dapaong_qc.csv')


# 1.  Summary Statistics
print(" Summary Statistics:")
print(df.describe())

# 2.  Missing Value Report
print("\n Missing Value Count:")
missing_values = df.isna().sum()
print(missing_values)

# 3.  Columns with >5% Missing Values
print("\nColumns with >5% Missing Values:")
total_rows = len(df)
threshold = total_rows * 0.05
high_null_cols = missing_values[missing_values > threshold]
print(high_null_cols)


In [ ]:

# 4.  Outlier Detection & Cleaning
key_columns = ['GHI', 'DNI', 'DHI', 'ModA', 'ModB', 'WS', 'WSgust']
existing_cols = [col for col in key_columns if col in df.columns]

# Compute Z-scores and identify outliers
z_scores = df[existing_cols].apply(zscore)
outliers = (np.abs(z_scores) > 3)

print("\n Outlier Count in Each Column:")
print(outliers.sum())

# Replace outliers with NaN
df_clean = df.copy()
df_clean.loc[outliers.any(axis=1), existing_cols] = np.nan

# Impute missing values in key columns with median
df_clean[existing_cols] = df_clean[existing_cols].fillna(df_clean[existing_cols].median())

# 5.  Export Cleaned Data
# Cleaned file will be saved to: data/benin_clean.csv (relative to working dir)
output_dir = "data"
output_path = os.path.join(output_dir, "benin_clean.csv")

os.makedirs(output_dir, exist_ok=True)
df_clean.to_csv(output_path, index=False)

print(f"\n Cleaned dataset saved to: {output_path}")


In [ ]:
import matplotlib.pyplot as plt

df['Timestamp'] = pd.to_datetime(df['Timestamp'])
plt.figure(figsize=(15, 5))
plt.plot(df['Timestamp'], df['GHI'], label='GHI')
plt.plot(df['Timestamp'], df['DNI'], label='DNI')
plt.plot(df['Timestamp'], df['DHI'], label='DHI')
plt.plot(df['Timestamp'], df['Tamb'], label='Tamb')
plt.title('GHI, DNI, DHI, Tamb Over Time')
plt.xlabel('Timestamp')
plt.ylabel('Value')
plt.legend()
plt.tight_layout()
plt.show()


In [ ]:
if 'Cleaning' in df.columns:
    df['Cleaning'] = df['Cleaning'].astype(str)
    df.groupby('Cleaning')[['ModA', 'ModB']].mean().plot(kind='bar')
    plt.title("Average ModA & ModB: Pre vs Post Cleaning")
    plt.ylabel("Irradiance")
    plt.xlabel("Cleaning Status")
    plt.xticks(rotation=0)
    plt.tight_layout()
    plt.show()


In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt

corr_cols = ['GHI', 'DNI', 'DHI', 'TModA', 'TModB']
plt.figure(figsize=(8, 6))
sns.heatmap(df[corr_cols].corr(), annot=True, cmap="coolwarm")
plt.title("📈 Correlation Heatmap")
plt.show()


In [ ]:
fig, axs = plt.subplots(2, 2, figsize=(12, 10))

axs[0, 0].scatter(df['WS'], df['GHI'], alpha=0.5)
axs[0, 0].set_title('WS vs GHI')

axs[0, 1].scatter(df['WSgust'], df['GHI'], alpha=0.5, color='orange')
axs[0, 1].set_title('WSgust vs GHI')

axs[1, 0].scatter(df['WD'], df['GHI'], alpha=0.5, color='green')
axs[1, 0].set_title('WD vs GHI')

axs[1, 1].scatter(df['RH'], df['Tamb'], alpha=0.5, color='red')
axs[1, 1].set_title('RH vs Tamb')

for ax in axs.flat:
    ax.set_xlabel("X")
    ax.set_ylabel("Y")

plt.tight_layout()
plt.show()


In [ ]:
import plotly.express as px

fig = px.bar_polar(df, r='WS', theta='WD', color='WS',
                   color_continuous_scale='Viridis',
                   title="Wind Rose: WS by Wind Direction")
fig.show()


In [ ]:

# Scatterplot to observe relationship between temperature and RH
plt.figure(figsize=(8, 6))
sns.scatterplot(data=df, x='RH', y='Tamb', alpha=0.6)
plt.title("Temperature vs. Relative Humidity")
plt.xlabel("Relative Humidity (%)")
plt.ylabel("Ambient Temperature (°C)")
plt.grid(True)
plt.show()


In [ ]:
# Bubble chart: GHI vs. Tamb, bubble size = RH or BP
plt.figure(figsize=(10, 7))
bubble_size = df['RH']  # You can switch to 'BP' for pressure

plt.scatter(df['Tamb'], df['GHI'], s=bubble_size, alpha=0.5, c=bubble_size, cmap='viridis')
plt.colorbar(label='Relative Humidity (%)')
plt.title("Bubble Chart: GHI vs. Tamb with RH as Bubble Size")
plt.xlabel("Ambient Temperature (°C)")
plt.ylabel("Global Horizontal Irradiance (W/m²)")
plt.grid(True)
plt.show()
